# Phase 1: creating a db of article metadata

The JSON metadata file is updated weekly by ArXiv directly on Kaggle: https://www.kaggle.com/datasets/Cornell-University/arxiv

In [ ]:
import duckdb
import json
import kagglehub
import os
from dotenv import load_dotenv
_ = load_dotenv(override=True)

In [2]:
# Download the dataset and get the path to the JSON file
dataset_dir = kagglehub.dataset_download("Cornell-University/arxiv")
json_path = os.path.join(dataset_dir, "arxiv-metadata-oai-snapshot.json")
print(json_path)  

100%|██████████| 1.67G/1.67G [00:37<00:00, 47.8MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/295/arxiv-metadata-oai-snapshot.json


In [3]:
# Read the first line of the JSON file we downloaded to get a better understanding of its structure
with open(json_path, 'r') as json_file:
    data = json.loads(json_file.readline())

print("Keys:", list(data.keys()))
print("\nSample values:")
data

Keys: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']

Sample values:


{'id': '0704.0001',
 'submitter': 'Pavel Nadolsky',
 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",
 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies',
 'comments': '37 pages, 15 figures; published version',
 'journal-ref': 'Phys.Rev.D76:013009,2007',
 'doi': '10.1103/PhysRevD.76.013009',
 'report-no': 'ANL-HEP-PR-07-12',
 'categories': 'hep-ph',
 'license': None,
 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from th

## DuckDB creation

In [4]:
conn = duckdb.connect('data/arxiv_metadata.duckdb')

In [5]:
conn.execute(f"""
    CREATE OR REPLACE TABLE staging AS
    SELECT * FROM read_json_auto('{json_path}');
""")

print(conn.execute("DESCRIBE staging").fetchdf())

print(conn.execute("SELECT * FROM staging LIMIT 1").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       column_name                                   column_type null   key  \
0               id                                       VARCHAR  YES  None   
1        submitter                                       VARCHAR  YES  None   
2          authors                                       VARCHAR  YES  None   
3            title                                       VARCHAR  YES  None   
4         comments                                       VARCHAR  YES  None   
5      journal-ref                                       VARCHAR  YES  None   
6              doi                                       VARCHAR  YES  None   
7        report-no                                       VARCHAR  YES  None   
8       categories                                       VARCHAR  YES  None   
9          license                                       VARCHAR  YES  None   
10        abstract                                       VARCHAR  YES  None   
11        versions  STRUCT("version" VARCHAR, create

In [6]:
# Check if there are duplicate arxiv_ids in the staging table
conn.execute("""
             SELECT id, COUNT(*) AS n
                FROM staging
                GROUP BY id
                HAVING COUNT(*) > 1
                ORDER BY n DESC;
             """).fetchdf()

,id,n
0,math-ph/0512019,4
1,math-ph/0605055,4
2,math-ph/0611044,3
3,math-ph/0203043,3
4,math-ph/0606001,3
...,...,...
73,math-ph/0010034,2
74,math-ph/0301028,2
75,math-ph/0609050,2
76,math-ph/0208029,2


In [7]:
# If there are duplicates, select the most recent one for each arxiv_id with QUALIFY
conn.execute("""
    CREATE OR REPLACE TABLE staging_clean AS
    SELECT *
    FROM staging
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY update_date DESC) = 1;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## TABLES

Let's create several tables that will allow us to naviguate through the metadata to get the papers (in PDF and / or LATEX formats) with the associated code.

**Architecture of the tables:**

papers: the table storing all raw informations from the ArXiv metadata
- arxiv_id (PK, TEXT)
- title, abstract (TEXT)
- submitted_date, updated_date (DATE)
- primary_category (TEXT, e.g. cs.LG)
- doi (TEXT, nullable)
- comments (TEXT, nullable — author's free-text note, often where GitHub URLs live)
- license (TEXT)
- version_count (INT)
- github_url (TEXT, nullable — extracted by your regex)
- has_code (BOOLEAN — derived: TRUE if github_url is not null)

paper_local: the table storing all informations about article fetching and OCR runs
- arxiv_id (PK, TEXT)
- pdf_path (TEXT)
- ocr_path (TEXT)
- ocr_model (VARCHAR)
- ocr_date_run (TIMESTAMP DEFAULT CURRENT_TIMESTAMP)
- keyword_for_ocr (VARCHAR)
- nb_pages_pdf (INTEGER)
- nb_pages_ocr (INTEGER)
- ocr_done (BOOLEAN)

ocr_stats: the table storing all informations about the OCR run
- arxiv_id (TEXT)
- page (INTEGER)
- n_tokens (INTEGER) 
- n_chars (INTEGER)
- n_images (INTEGER)
- error (VARCHAR)

authors: the table storing all informations about authors
- author_id (INTEGER, auto)
- name_raw (TEXT)
- name_normalized (TEXT — e.g. lowercased, accents stripped — for joining)

paper_authors: the table storing positions of all authors across all papers
- arxiv_id (FK), 
- author_id (FK), 
- position (INT)
- PK is (arxiv_id, position)

paper_categories: table storing the categories of each paper
- arxiv_id (FK), 
- category (TEXT), 
- is_primary (BOOLEAN)

ingestion_runs: a table to store information about updates
- run_id (auto), 
- source ('kaggle_dump' or 'arxiv_api'), 
- started_at, 
- completed_at, 
- n_papers, 
- n_files, 
- status, 
- notes

In [8]:
# First table: papers 
conn.execute("""
CREATE OR REPLACE TABLE papers (
             arxiv_id VARCHAR PRIMARY KEY,
             title VARCHAR,
             abstract VARCHAR,
             submitted_date DATE, 
             updated_date DATE,
             primary_category VARCHAR,
             doi VARCHAR,
             comments VARCHAR,
             license VARCHAR,
             version_count BIGINT,
             github_url VARCHAR,
             has_code BOOLEAN
             );
             """)

# Second: add values from staging_clean to the table papers
conn.execute(r"""
INSERT INTO papers
SELECT
    id AS arxiv_id,
    title,
    abstract,

    -- versions is a LIST of STRUCTs like [{version: 'v1', created: 'Mon, 1 Jan 2023 ...'}, ...]
    -- DuckDB lists are 1-indexed (NOT 0-indexed like Python).
    -- The 'created' field is in RFC 2822 format, so we parse it with strptime.
    -- We treat 'GMT' as a literal because timezone parsing is finicky.
    strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT')::DATE
        AS submitted_date,

    -- update_date is already in YYYY-MM-DD format, just cast it
    update_date::DATE AS updated_date,

    -- categories is a space-separated string like 'cs.LG cs.AI stat.ML'
    -- split_part(text, ' ', 1) returns the FIRST piece
    split_part(categories, ' ', 1) AS primary_category,

    doi,
    comments,
    license,

    -- length() on a list returns its size
    length(versions) AS version_count,

    -- GitHub URL extraction:
    NULLIF(
        regexp_extract(
            COALESCE(abstract, '') || ' ' || COALESCE(comments, ''),
            'https?://github\.com/[\w\-\.]+/[\w\-\.]+'
        ),
        ''
    ) AS github_url,

    -- Derived boolean: TRUE if we found a GitHub link above.
    -- We compute it inline rather than as a generated column for portability.
    (NULLIF(
        regexp_extract(
            COALESCE(abstract, '') || ' ' || COALESCE(comments, ''),
            'https?://github\.com/[\w\-\.]+/[\w\-\.]+'
        ),
        ''
    ) IS NOT NULL) AS has_code


FROM staging_clean;
""")

# How many papers with code ? 
conn.execute("SELECT COUNT(*) AS n_papers, COUNT(github_url) AS n_with_code FROM papers").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_papers,n_with_code
0,3106926,77792


In [9]:
# Create paper_local to store locally all informations about the papers, including the paths to the downloaded PDFs and LaTeX sources.
conn.execute("""
                 CREATE TABLE IF NOT EXISTS paper_local(
                 arxiv_id VARCHAR PRIMARY KEY,
                 pdf_path VARCHAR,
                 latex_source_path VARCHAR,
                 latex_extracted BOOLEAN,
                 md_path VARCHAR,
                 ocr_model VARCHAR,
                 ocr_date TIMESTAMP
                 );
                 """)

# Insert the data in the new table
#conn.execute("""
#INSERT INTO paper_local (arxiv_id, pdf_path, latex_source_path, latex_extracted)
#SELECT arxiv_id, pdf_path, latex_source_path, latex_extracted
#FROM papers                    
#WHERE pdf_path IS NOT NULL;
#             """)       


In [10]:
# Add some columns to paper_local for storing ocr runs informations
conn.execute("ALTER TABLE paper_local ADD COLUMN IF NOT EXISTS ocr_path VARCHAR;")
conn.execute("ALTER TABLE paper_local ADD COLUMN IF NOT EXISTS keyword_for_ocr VARCHAR;")
conn.execute("ALTER TABLE paper_local ADD COLUMN IF NOT EXISTS nb_pages_pdf INTEGER;")
conn.execute("ALTER TABLE paper_local ADD COLUMN IF NOT EXISTS nb_pages_ocr INTEGER;")
conn.execute("ALTER TABLE paper_local ADD COLUMN IF NOT EXISTS ocr_done BOOLEAN DEFAULT FALSE;")
conn.execute("ALTER TABLE paper_local DROP COLUMN IF EXISTS md_path;")

In [11]:
conn.execute("SELECT COUNT(*) FROM paper_local").fetchdf()

,count_star()
0,3074


In [12]:
# ocr_stats
conn.execute("""
CREATE TABLE IF NOT EXISTS ocr_stats(
    arxiv_id VARCHAR,
    page INTEGER,
    n_tokens INTEGER,
    n_chars INTEGER,
    n_images INTEGER,
    error VARCHAR
);
""")

In [13]:
# papers_categories
conn.execute("""
CREATE OR REPLACE TABLE paper_categories (
            arxiv_id VARCHAR,
            category VARCHAR,
            is_primary BOOLEAN,
            PRIMARY KEY (arxiv_id, category)
);
""")

conn.execute("""
INSERT INTO paper_categories
SELECT DISTINCT 
    arxiv_id,
    category,
    (category = primary_category) AS is_primary
FROM (
    SELECT
        id AS arxiv_id,
        UNNEST (string_split(categories, ' ')) AS category,
        string_split(categories, ' ')[1] AS primary_category
    FROM staging_clean           
);
""")

conn.execute("""
SELECT category, COUNT(*) AS n_papers
FROM paper_categories
GROUP BY category
ORDER BY n_papers DESC
LIMIT 50
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category,n_papers
0,cs.LG,277439
1,cs.CV,199382
2,hep-ph,197683
3,cs.AI,189708
4,hep-th,183994
5,quant-ph,182859
6,gr-qc,123150
7,cs.CL,114362
8,cond-mat.mtrl-sci,110538
9,astro-ph,105380


In [14]:
# Step 1: explode authors_parsed with their position in the author list
conn.execute("""
CREATE OR REPLACE TABLE _exploded AS
SELECT
    arxiv_id,
    -- author_arr is a list like ['Smith', 'John P.', '']  (last, first, suffix)
    -- array_to_string joins it with spaces: 'Smith John P. '
    -- TRIM removes the trailing space
    TRIM(array_to_string(author_arr, ' ')) AS name_raw,
    -- Normalized form for joining: lowercase + accents stripped
    -- 'François' -> 'francois', so 'François Chollet' matches 'Francois Chollet'
    LOWER(strip_accents(TRIM(array_to_string(author_arr, ' ')))) AS name_normalized,
    position
FROM (
    SELECT
        id AS arxiv_id,
        author_arr,
        -- ROW_NUMBER assigns 1, 2, 3... within each paper, preserving UNNEST order
        ROW_NUMBER() OVER (PARTITION BY id) AS position
    FROM (
        SELECT id, UNNEST(authors_parsed) AS author_arr FROM staging_clean
    )
);
""")

# Step 2: deduplicated authors table
conn.execute("""
CREATE OR REPLACE TABLE authors (
    author_id       BIGINT  PRIMARY KEY,        
    name_raw        VARCHAR,
    name_normalized VARCHAR UNIQUE          
);
INSERT INTO authors             
SELECT
    ROW_NUMBER() OVER () AS author_id,
    name_raw,
    name_normalized
FROM (
    -- For each unique normalized name, pick any one of its raw spellings
    SELECT
        FIRST(name_raw) AS name_raw,
        name_normalized
    FROM _exploded
    GROUP BY name_normalized
);
""")

# Step 3: the link table
conn.execute("""
             CREATE OR REPLACE TABLE paper_authors (
    arxiv_id  VARCHAR,
    author_id BIGINT,
    position  INTEGER,
    PRIMARY KEY (arxiv_id, position)   -- « l'auteur à la place n°N de ce papier »
);
             """)

conn.execute("""
INSERT INTO paper_authors
SELECT
    e.arxiv_id,
    a.author_id,
    e.position
FROM _exploded e
JOIN authors a USING (name_normalized);
""")

# Drop the temp scratch table
conn.execute("DROP TABLE _exploded;")

# Sanity check: top 5 most prolific authors in the corpus
conn.execute("""
SELECT a.name_raw, COUNT(*) AS n_papers
FROM paper_authors pa
JOIN authors a USING (author_id)
GROUP BY a.name_raw
ORDER BY n_papers DESC
LIMIT 5
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name_raw,n_papers
0,Zhang Y.,3403
1,Liu Yang,2955
2,Wang J.,2566
3,Wang Y.,2542
4,Wang Wei,2414


In [15]:
# Create a chunk table, empty now but will be used for RAG
conn.execute("""CREATE TABLE IF NOT EXISTS chunks (
    chunk_id    BIGINT PRIMARY KEY,
    arxiv_id    VARCHAR,
    chunk_index INTEGER,
    char_start  INTEGER,
    char_end    INTEGER,
    n_tokens    INTEGER,
    text        VARCHAR,
    section     VARCHAR,
    embed_model VARCHAR,
    UNIQUE (arxiv_id, chunk_index)
);
""")

In [16]:
conn.execute("""
SELECT COUNT(*) FROM paper_categories pc
  LEFT JOIN papers p USING (arxiv_id) WHERE p.arxiv_id IS NULL;
             """).fetchdf()

,count_star()
0,0


In [17]:
# Une séquence = un compteur. nextval() avance le compteur et renvoie le nouveau nombre.
conn.execute("CREATE SEQUENCE IF NOT EXISTS seq_run_id START 1;")

# ingestion_runs
conn.execute("""
CREATE TABLE IF NOT EXISTS ingestion_runs (
    run_id       INTEGER DEFAULT nextval('seq_run_id') PRIMARY KEY,
    source       VARCHAR,
    started_at   TIMESTAMP,
    completed_at TIMESTAMP,
    n_papers     INTEGER,
    n_files      INTEGER,
    watermark    DATE,                                             
    status       VARCHAR,
    notes        VARCHAR
);
""")

In [18]:
conn.execute("""
INSERT INTO ingestion_runs (
    source, started_at, completed_at, n_papers, n_files, watermark, status, notes
)
SELECT
    'kaggle_dump',
    CURRENT_TIMESTAMP,
    CURRENT_TIMESTAMP,
    (SELECT COUNT(*) FROM papers),
    0,
    (SELECT MAX(updated_date) FROM papers),
    'success',
    'Initial bulk load from Kaggle JSONL snapshot.'
""")

conn.execute("SELECT * FROM ingestion_runs").fetchdf()

,run_id,source,started_at,completed_at,n_papers,n_files,watermark,status,notes
0,1,kaggle_dump,2026-06-08 18:45:26.788978,2026-06-08 18:45:26.788978,3066102,0,2026-06-05,success,Initial bulk load from Kaggle JSONL snapshot.
1,2,kaggle_dump,2026-06-15 20:42:11.889130,2026-06-15 20:42:11.889130,3073288,0,2026-06-13,success,Initial bulk load from Kaggle JSONL snapshot.
2,3,kaggle_dump,2026-07-22 16:05:51.340590,2026-07-22 16:05:51.340590,3106926,0,2026-07-17,success,Initial bulk load from Kaggle JSONL snapshot.


In [19]:
# Show all tables
conn.execute("SHOW TABLES").fetchdf()

# Check the size of staging vs the normalized tables
conn.execute("""
SELECT 'staging' AS tbl, COUNT(*) AS n FROM staging
UNION ALL SELECT 'papers', COUNT(*) FROM papers
UNION ALL SELECT 'paper_categories', COUNT(*) FROM paper_categories
UNION ALL SELECT 'authors', COUNT(*) FROM authors
UNION ALL SELECT 'paper_authors', COUNT(*) FROM paper_authors
""").fetchdf()

,tbl,n
0,staging,3107014
1,papers,3106926
2,paper_categories,5391826
3,authors,2393008
4,paper_authors,14760264


In [20]:
# Cleanup: drop staging tables to free up space
conn.execute("DROP TABLE IF EXISTS staging;")
conn.execute("DROP TABLE IF EXISTS staging_clean;")
conn.execute("CHECKPOINT;") 

In [21]:
# close the db:
conn.close()

## Input list for fetching PDFs

The goal is to query our db to get 1000 recent papers on computer science for OCR and benchmarking the different open source algorithms to do so. 

In [22]:
conn = duckdb.connect('data/arxiv_metadata.duckdb')

# Let's verify the distribution of papers across CS
conn.execute("""
            SELECT primary_category, COUNT(*) AS n_papers
            FROM papers
            WHERE primary_category LIKE 'cs.%' AND submitted_date >= '2024-01-01'
            GROUP BY primary_category
            ORDER BY n_papers DESC
""").fetchdf()

,primary_category,n_papers
0,cs.CV,69554
1,cs.LG,63792
2,cs.CL,42362
3,cs.AI,22178
4,cs.RO,21019
5,cs.CR,14294
6,cs.HC,10444
7,cs.SE,9981
8,cs.IR,6639
9,cs.CY,6426


First, let's acknolewdge the work of all these amazing scientists publishing so many papers in only 2 years, it is pretty impressive!

In [23]:
# Let's select the same number of papers across all 10 categories of CS:
conn.execute("""
             CREATE OR REPLACE TABLE benchmark_subset AS
             WITH ranked AS (
                 SELECT
                     arxiv_id,
                     title,
                     primary_category,
                     submitted_date,
                     has_code,
                     github_url,
                     ROW_NUMBER() OVER (
                         PARTITION BY primary_category
                         ORDER BY HASH(arxiv_id || '_seed42')
                     ) AS rn
                 FROM papers
                 WHERE primary_category IN (
                     'cs.CV', 'cs.LG', 'cs.CL', 'cs.RO', 'cs.AI',
                     'cs.CR', 'cs.HC', 'cs.SE', 'cs.IT', 'cs.IR'
                 )
                 AND submitted_date >= '2024-01-01'
                 AND submitted_date <= '2026-01-01'
             )
             SELECT
                 arxiv_id,
                 title,
                 primary_category,
                 submitted_date,
                 has_code,
                 github_url
             FROM ranked
             WHERE rn <= 100
             ORDER BY primary_category, rn;
             """)

conn.execute("""
SELECT primary_category, COUNT(*) AS n,
       SUM(CASE WHEN has_code THEN 1 ELSE 0 END) AS n_with_code
FROM benchmark_subset
GROUP BY primary_category
ORDER BY primary_category
""").fetchdf()

,primary_category,n,n_with_code
0,cs.AI,100,13.0
1,cs.CL,100,13.0
2,cs.CR,100,4.0
3,cs.CV,100,23.0
4,cs.HC,100,0.0
5,cs.IR,100,18.0
6,cs.IT,100,0.0
7,cs.LG,100,11.0
8,cs.RO,100,5.0
9,cs.SE,100,3.0


In [24]:
conn.execute("""
    SELECT *
    FROM benchmark_subset
    LIMIT 30
""").fetchdf()

,arxiv_id,title,primary_category,submitted_date,has_code,github_url
0,2510.25612,Counterfactual-based Agent Influence Ranker fo...,cs.AI,2025-10-29,False,None
1,2508.01324,Towards Evaluation for Real-World LLM Unlearning,cs.AI,2025-08-02,False,None
2,2506.12483,MALM: A Multi-Information Adapter for Large La...,cs.AI,2025-06-14,False,None
3,2510.11143,Spec-Driven AI for Science: The ARIA Framework...,cs.AI,2025-10-13,False,None
4,2412.13422,Generating Diverse Hypotheses for Inductive Re...,cs.AI,2024-12-18,False,None
5,2410.01290,Towards a Law of Iterated Expectations for Heu...,cs.AI,2024-10-02,False,None
6,2501.08074,Artificial Liver Classifier: A New Alternative...,cs.AI,2025-01-14,False,None
7,2509.13347,OpenHA: A Series of Open-Source Hierarchical A...,cs.AI,2025-09-13,True,https://github.com/CraftJarvis/OpenHA
8,2512.16030,Do Large Language Models Know What They Don't ...,cs.AI,2025-12-17,False,None
9,2510.17052,ToolCritic: Detecting and Correcting Tool-Use ...,cs.AI,2025-10-19,False,None


In [25]:
conn.execute("""
    SELECT *
    FROM papers
    LIMIT 10
""").fetchdf()

,arxiv_id,title,abstract,submitted_date,updated_date,primary_category,doi,comments,license,version_count,github_url,has_code
0,math/0403376,The Kunneth formula in Floer homology for mani...,We prove the Kunneth formula in Floer (co)ho...,2004-03-22,2007-05-23,math.SG,10.1007/s00208-005-0700-0,"21 pages, 3 figures. Major reorganization and ...",None,2,None,False
1,math/0403377,A survey of Floer homology for manifolds with ...,The purpose of this paper is to give a surve...,2004-03-22,2007-05-23,math.SG,None,"32 pages, 5 figures",None,1,None,False
2,math/0403378,On the Constructive Inverse Problem in Differe...,We give sufficient conditions for a linear d...,2004-03-23,2007-05-23,math.GM,None,Several misprints have been corrected and the ...,None,3,None,False
3,math/0403379,Toric degenerations of spherical varieties,"We prove that any affine, resp. polarized pr...",2004-03-23,2007-05-23,math.AG,None,"22 pages, 1 figure",None,1,None,False
4,math/0403380,Generalized $C^1$ quadratic B-splines generate...,A new global basis of B-splines is defined i...,2004-03-23,2025-10-20,math.NA,None,2004-13,None,1,None,False
5,math/0403381,Constant mean curvature surfaces of any positi...,We show the existence of several new familie...,2004-03-23,2007-05-23,math.DG,10.1112/S000246107050006472,"14 pages, 10 figures",None,1,None,False
6,math/0403382,Divisorial contractions of 3-folds,In this paper the three-dimensional divisori...,2004-03-23,2014-11-24,math.AG,None,This paper has been withdrawn by the author du...,None,3,None,False
7,math/0403383,Invariant tubular neighborhood theorem for aff...,The aim of this note is to prove the algebra...,2004-03-23,2007-05-23,math.RT,None,6 pages,None,1,None,False
8,math/0403384,On the Hartogs-Bochner phenomenon for CR funct...,"Let M be a compact, connected, C^2-smooth an...",2004-03-23,2009-09-29,math.CV,None,"7 pages, 1 figure",None,1,None,False
9,math/0403385,Exact convergence rates in the central limit t...,We give optimal convergence rates in the cen...,2004-03-23,2007-05-23,math.PR,None,Soumis pour publication,None,1,None,False


In [26]:
# Final check of what has been downloaded after running fetch_files.py
conn.execute("""
    SELECT
        SUM(CASE WHEN pdf_path IS NOT NULL THEN 1 ELSE 0 END) AS done,
        SUM(CASE WHEN pdf_path IS NULL THEN 1 ELSE 0 END) AS remaining
    FROM benchmark_subset bs JOIN papers p USING (arxiv_id)
""").fetchdf()

BinderException: Binder Error: Referenced column "pdf_path" not found in FROM clause!
Candidate bindings: "updated_date", "bs.primary_category", "p.primary_category"

LINE 3:         SUM(CASE WHEN pdf_path IS NOT NULL THEN 1 ELSE 0 END) AS done,
                              ^

In [27]:
# After use, close the connection to our duckdb to avoid locking issues in future runs
conn.close()

Testing the db for upsert

In [28]:
import duckdb
conn = duckdb.connect("data/arxiv_metadata.duckdb")
print(conn.execute("DESCRIBE papers").fetchdf())

         column_name column_type null   key default extra
0           arxiv_id     VARCHAR   NO   PRI    None  None
1              title     VARCHAR  YES  None    None  None
2           abstract     VARCHAR  YES  None    None  None
3     submitted_date        DATE  YES  None    None  None
4       updated_date        DATE  YES  None    None  None
5   primary_category     VARCHAR  YES  None    None  None
6                doi     VARCHAR  YES  None    None  None
7           comments     VARCHAR  YES  None    None  None
8            license     VARCHAR  YES  None    None  None
9      version_count      BIGINT  YES  None    None  None
10        github_url     VARCHAR  YES  None    None  None
11          has_code     BOOLEAN  YES  None    None  None


In [29]:
conn.close()

# Selecting a scientific topic to launch OCR

Use these lines of code to create a custom table where papers related to your topic of interest will be selected from the entire ArXiv to be OCR'd.

In [2]:
import duckdb
import json
import kagglehub
import os
from dotenv import load_dotenv
_ = load_dotenv(override=True)

In [3]:
conn = duckdb.connect('data/arxiv_metadata.duckdb')

In [22]:
conn.execute(r"""
CREATE OR REPLACE TABLE genai_subset AS
SELECT arxiv_id, title, primary_category, submitted_date, has_code, github_url
FROM papers
WHERE primary_category IN ('cs.CL','cs.LG','cs.CV','cs.AI')
  AND submitted_date >= '2025-01-01'
  AND regexp_matches(lower(title || ' ' || coalesce(abstract,'')),
        'generative|large language model|\bllm\b|\bgpt\b|transformer|diffusion model|text-to-image|foundation model|retrieval.augmented|multimodal|instruction.tun|prompt|fine.tun|\bagent')
ORDER BY submitted_date DESC
LIMIT 7000;
""").fetchdf()

,Count
0,7000


In [23]:
conn.execute("""
SELECT * FROM genai_subset
ORDER BY submitted_date DESC
LIMIT 10;
""").fetchdf()

,arxiv_id,title,primary_category,submitted_date,has_code,github_url
0,2606.12997,Reliability of Probabilistic Emulation of Phys...,cs.LG,2026-06-11,False,None
1,2606.13003,The Illusion of Multi-Agent Advantage,cs.AI,2026-06-11,False,None
2,2606.13007,scLLM-DSC: LLM-Knowledge Enhanced Cross-Modal ...,cs.LG,2026-06-11,False,None
3,2606.13016,Otters++: A Time-to-first-spike Based Energy E...,cs.AI,2026-06-11,False,None
4,2606.13024,CausalMoE: A Billion-Scale Multimodal Foundati...,cs.LG,2026-06-11,False,None
5,2606.13044,No Hidden Prompts Needed! You Can Game AI Peer...,cs.CL,2026-06-11,False,None
6,2606.13051,AAbAAC: An Annotated Corpus for Autoimmunity I...,cs.AI,2026-06-11,True,https://github.com/f-maury/AAbAAC.
7,2606.13054,TWLA: Achieving Ternary Weights and Low-Bit Ac...,cs.LG,2026-06-11,True,https://github.com/Kishon-zzx/TWLA
8,2606.13060,A green solvent screening tool for emerging ma...,cs.LG,2026-06-11,False,None
9,2606.13104,"Authority, Truth, and Citation Bias: A Large-S...",cs.LG,2026-06-11,True,https://github.com/floating-reeds/AuthorityBench


In [24]:
# After use, close the connection to our duckdb to avoid locking issues in future runs
conn.close()

# Creates table for topic retrieval

Regex pulls too many papers that are not related to the task so we will use a hybrid search combined with a LLM classification to only select relevant papers to topic of interest

In [30]:
import duckdb

conn = duckdb.connect("data/arxiv_metadata.duckdb")
conn.execute("""
    CREATE TABLE IF NOT EXISTS abstract_embeddings (
        arxiv_id       VARCHAR PRIMARY KEY,
        dense_vec      FLOAT[1024],
        sparse_weights MAP(INTEGER, FLOAT),
        embed_model    VARCHAR,
        embed_date     TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.execute("CREATE SEQUENCE IF NOT EXISTS seq_topic_run_id START 1")
conn.execute("""
    CREATE TABLE IF NOT EXISTS topic_subset (
        topic         VARCHAR,
        arxiv_id      VARCHAR,
        hybrid_score  DOUBLE,
        llm_verdict   BOOLEAN,
        rank          INTEGER,
        run_id        INTEGER,
        created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (topic, arxiv_id)
    )
""")
print(conn.execute("SHOW TABLES").fetchdf())
conn.close()

                   name
0   abstract_embeddings
1               authors
2      benchmark_subset
3                chunks
4          genai_subset
5        ingestion_runs
6             ocr_stats
7         paper_authors
8      paper_categories
9           paper_local
10               papers
11         topic_subset
